# 🥐 **연관 분석 실습 : 베이커리 카페 장바구니 뜯어보기**

<span style="color:black; background-color:#E6E6FA; padding:2px 4px; border-radius:4px">
<strong> 🤓 이번 실습은 <u>코드를 채워 넣는 실습이 아닙니다!</u> <strong>
</span>

이번 실습의 코드는 **전부 완성된 상태**로 제공됩니다.
여러분이 할 일은 딱 두 가지예요.

1. **셀을 순서대로 실행**한다.
2. **코드가 무엇을 하고 있는지 / 출력된 숫자가 무엇을 뜻하는지 직접 문장으로 적는다.**

중간중간 <span style="color:black; background-color:#FFD6E0; padding:2px 4px; border-radius:4px"><strong> ✍️ Q <strong></span> 표시가 있는 셀이 나옵니다.
해당 셀을 더블클릭해서 `>` 뒤에 자신의 답을 작성해주세요.

<span style="color:gray">💡 정답을 맞히는 것보다, <b>"왜 이 코드가 필요한가"</b>를 스스로 설명해보는 게 목적입니다.
숫자를 그대로 옮겨 적기보다 <b>"그래서 사장님한테 뭐라고 말할 건데?"</b>까지 써보세요.</span>

### <span style="color:purple"> <strong> 0. 준비하기 <strong> </span>

In [1]:
!pip install mlxtend --quiet


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import warnings
warnings.filterwarnings("ignore")

import time
import pandas as pd

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)

## **1. 데이터 살펴보기**

### <span style="color:orange"> <strong> 1-1. 데이터 불러오기 <strong> </span>

#### 🍞 오늘의 데이터: **The Bread Basket**

영국 에든버러 구시가지에 있는 실제 베이커리 카페의 판매 기록입니다.
2016년 10월 ~ 2017년 4월 사이의 거래가 담겨 있어요.

| 컬럼 | 설명 |
| --- | --- |
| `Date` | 거래 날짜 |
| `Time` | 거래 시각 |
| `Transaction` | **거래(영수증) 번호** — 같은 번호끼리는 한 장바구니 |
| `Item` | 구매한 품목 |

In [3]:
# 데이터 불러오기 (인터넷 연결이 안 되면 아래 주석 처리된 줄을 사용하세요)
url = "https://raw.githubusercontent.com/victoriavc/Association-Rules-for-Market-Basket-Analysis/master/BreadBasket_DMS.csv"
bakery = pd.read_csv(url)
# bakery = pd.read_csv('bakery.csv')

bakery.head(10)

,Date,Time,Transaction,Item
0,2016-10-30,09:58:11,1,Bread
1,2016-10-30,10:05:34,2,Scandinavian
2,2016-10-30,10:05:34,2,Scandinavian
3,2016-10-30,10:07:57,3,Hot chocolate
4,2016-10-30,10:07:57,3,Jam
5,2016-10-30,10:07:57,3,Cookies
6,2016-10-30,10:08:41,4,Muffin
7,2016-10-30,10:13:03,5,Coffee
8,2016-10-30,10:13:03,5,Pastry
9,2016-10-30,10:13:03,5,Bread


In [4]:
# 데이터의 크기와 구성 확인
print("전체 행(row) 개수 :", len(bakery))
print("고유한 거래 번호 개수 :", bakery['Transaction'].nunique())
print("고유한 품목 개수 :", bakery['Item'].nunique())

전체 행(row) 개수 : 21293
고유한 거래 번호 개수 : 9531
고유한 품목 개수 : 95


<span style="color:black; background-color:#FFD6E0; padding:2px 4px; border-radius:4px">
<strong> ✍️ Q1. 행의 개수와 거래 번호의 개수가 다릅니다. 왜 그럴까요? <strong>
</span>

위 출력에서 **전체 행 개수**와 **고유한 거래 번호 개수**를 비교해보세요.
그리고 맨 위 `head(10)` 출력에서 `Transaction` 컬럼이 **2, 2** 처럼 반복되는 부분을 다시 확인해보세요.

- 이 데이터에서 **한 행(row)** 은 무엇을 의미하나요?
- 이 데이터에서 **한 장바구니**는 어떻게 표현되어 있나요?


---

**📝 나의 답변**

>
>
>
>
>
>

### <span style="color:orange"> <strong> 1-2. 이상한 품목 찾아내기 <strong> </span>

<span style="color:black; background-color:#FFF099; padding:2px 4px; border-radius:4px">
<strong> 🤓 실제 데이터에는 항상 '이상한 값'이 섞여 있습니다. 품목 목록을 한 번 훑어볼까요? <strong>
</span>

In [5]:
# 가장 많이 팔린 품목 상위 15개
bakery['Item'].value_counts().head(15)

Item
Coffee           5471
Bread            3325
Tea              1435
Cake             1025
Pastry            856
NONE              786
Sandwich          771
Medialuna         616
Hot chocolate     590
Cookies           540
Brownie           379
Farm House        374
Muffin            370
Juice             369
Alfajores         369
Name: count, dtype: int64

In [6]:
# 'NONE' 값 제거
print("제거 전 행 개수 :", len(bakery))

bakery = bakery[bakery['Item'] != 'NONE']

print("제거 후 행 개수 :", len(bakery))
print("제거된 행 개수 :", 21293 - len(bakery))

제거 전 행 개수 : 21293
제거 후 행 개수 : 20507
제거된 행 개수 : 786


<span style="color:black; background-color:#FFD6E0; padding:2px 4px; border-radius:4px">
<strong> ✍️ Q2. `NONE`은 왜 제거해야 할까요? <strong>
</span>

상위 15개 품목 목록 안에 `NONE`이 6위로 들어가 있었습니다.

- `NONE`을 제거하지 않고 그대로 분석하면 어떤 문제가 생길까요?
- 힌트: `NONE`은 전체 거래의 상당 부분에 등장합니다. 만약 이걸 하나의 '상품'으로 취급한다면,
  `NONE → Coffee` 같은 규칙이 튀어나올 수도 있겠죠. 그 규칙은 의미가 있을까요?

---

**📝 나의 답변**

>
>
>
>
>
>

## **2. 트랜잭션 데이터로 변환하기**

### <span style="color:orange"> <strong> 2-1. 세로로 긴 데이터 → 장바구니 리스트 <strong> </span>

<span style="color:black; background-color:#FFF099; padding:2px 4px; border-radius:4px">
<strong> 🤓 지금 데이터는 '한 행 = 한 품목' 형태입니다. 하지만 연관 분석은 '한 행 = 한 장바구니'를 원해요! <strong>
</span>

```
[지금 모습]                          [원하는 모습]
Transaction   Item                  
    3         Hot chocolate    →     ['Hot chocolate', 'Jam', 'Cookies']
    3         Jam              →     
    3         Cookies          →     
```

In [7]:
# Transaction 번호로 묶어서, 같은 영수증의 품목들을 하나의 리스트로 만들기
transactions = (
    bakery
    .groupby('Transaction')['Item']      # 거래번호별로 묶고
    .apply(lambda s: sorted(set(s)))     # 중복 제거 후 정렬해서 리스트로
    .tolist()                            # 파이썬 리스트로 변환
)

print("총 장바구니 개수 :", len(transactions))
print()
print("앞에서 5개만 미리보기")
for t in transactions[:5]:
    print(" ", t)

총 장바구니 개수 : 9465

앞에서 5개만 미리보기
  ['Bread']
  ['Scandinavian']
  ['Cookies', 'Hot chocolate', 'Jam']
  ['Muffin']
  ['Bread', 'Coffee', 'Pastry']


<span style="color:black; background-color:#FFD6E0; padding:2px 4px; border-radius:4px">
<strong> ✍️ Q3. `sorted(set(s))`에서 `set()`은 무슨 일을 하고 있나요? <strong>
</span>

맨 처음 `head(10)` 출력을 보면, 2번 거래에서 `Scandinavian`이 **두 번** 기록되어 있었습니다.
(= 손님이 스칸디나비안을 2개 샀다는 뜻이죠)

- 연관 분석에서 **"몇 개 샀는지"** 보다 **"샀는지 안 샀는지"** 만 보는 이유는 무엇일까요?
- 만약 `set()`을 빼고 실행한다면 지지도(support) 계산이 어떻게 왜곡될까요?

---

**📝 나의 답변**

>
>
>
>
>
>

In [8]:
# 장바구니 하나에 평균 몇 개의 품목이 담겼을까?
basket_size = pd.Series([len(t) for t in transactions])
basket_size.describe()

count    9465.000000
mean        1.995457
std         1.129543
min         1.000000
25%         1.000000
50%         2.000000
75%         3.000000
max        10.000000
dtype: float64

### <span style="color:orange"> <strong> 2-2. TransactionEncoder로 이진 행렬 만들기 <strong> </span>

- 알고리즘은 **모든 거래를 동일한 기준(전체 상품 목록)** 위에서 비교해야 합니다.
- 그래서 `TransactionEncoder`로 **행 = 장바구니 / 열 = 상품 / 값 = True·False** 인 이진 행렬로 바꿔줍니다.

In [9]:
te = TransactionEncoder()
te_ary = te.fit_transform(transactions)

basket_df = pd.DataFrame(te_ary, columns=te.columns_)

print("행렬 크기 (행, 열) :", basket_df.shape)
basket_df.head()

행렬 크기 (행, 열) : (9465, 94)


,Adjustment,Afternoon with the baker,Alfajores,Argentina Night,Art Tray,Bacon,Baguette,Bakewell,Bare Popcorn,Basket,...,The BART,The Nomad,Tiffin,Toast,Truffles,Tshirt,Valentine's card,Vegan Feast,Vegan mincepie,Victorian Sponge
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [10]:
# 앞에서 본 3번 장바구니(['Cookies', 'Hot chocolate', 'Jam'])가
# 이진 행렬에서 어떻게 표현되었는지 직접 확인해보기
basket_df.loc[2, ['Cookies', 'Hot chocolate', 'Jam', 'Coffee', 'Bread']]

Cookies           True
Hot chocolate     True
Jam               True
Coffee           False
Bread            False
Name: 2, dtype: bool

<span style="color:black; background-color:#FFD6E0; padding:2px 4px; border-radius:4px">
<strong> ✍️ Q4. 이진 행렬의 행과 열은 각각 무엇을 의미하나요? <strong>
</span>

`basket_df.shape`로 출력된 두 숫자를 각각 설명해보세요.

- 행(row)의 개수는 무엇을 뜻하나요?
- 열(column)의 개수는 무엇을 뜻하나요?
- `basket_df['Coffee'].mean()` 을 계산하면 어떤 값이 나올까요? 
  → 아래 빈 셀에서 직접 실행해보고 확인해보세요!

---

**📝 나의 답변**

>
>
>
>
>
>

In [ ]:
# 👆 Q4를 직접 확인해보는 셀입니다. 


## **3. Apriori로 빈발 항목 집합 찾기**

### <span style="color:orange"> <strong> 3-1. 최소 지지도를 넘는 조합 추출 <strong> </span>

In [12]:
# min_support=0.02 → 전체 장바구니의 2% 이상에서 등장한 조합만 남긴다
frequent_itemsets = apriori(basket_df, min_support=0.02, use_colnames=True)

# 조합에 들어있는 품목 개수를 'length' 컬럼으로 추가
frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(len)

print("추출된 빈발 항목 집합 개수 :", len(frequent_itemsets))
frequent_itemsets.sort_values('support', ascending=False).head(15)

추출된 빈발 항목 집합 개수 : 33


,support,itemsets,length
4,0.478394,(Coffee),1
1,0.327205,(Bread),1
16,0.142631,(Tea),1
3,0.103856,(Cake),1
20,0.090016,"(Bread, Coffee)",2
11,0.086107,(Pastry),1
12,0.071844,(Sandwich),1
9,0.061807,(Medialuna),1
7,0.058320,(Hot chocolate),1
23,0.054728,"(Cake, Coffee)",2


<span style="color:black; background-color:#FFD6E0; padding:2px 4px; border-radius:4px">
<strong> ✍️ Q5. 위 표의 1등과, 2개짜리 조합 중 1등을 각각 해석해보세요. <strong>
</span>

- **1위 항목의 support 값**을 보고, 이 가게의 특성을 한 문장으로 표현해보세요.
- 아래 셀을 실행해 **2개짜리 조합** 중 1위를 확인하고, 그 support 값이 무엇을 의미하는지
  `"전체 ○○개의 장바구니 중 약 ○○%에서 ~~"` 형태의 문장으로 써보세요.

---

**📝 나의 답변**

>
>
>
>
>
>

In [13]:
# 품목이 2개 이상인 조합만 골라서 보기
frequent_itemsets[frequent_itemsets['length'] >= 2] \
    .sort_values('support', ascending=False)

,support,itemsets,length
20,0.090016,"(Bread, Coffee)",2
23,0.054728,"(Cake, Coffee)",2
31,0.049868,"(Coffee, Tea)",2
29,0.047544,"(Coffee, Pastry)",2
30,0.038246,"(Coffee, Sandwich)",2
28,0.035182,"(Coffee, Medialuna)",2
26,0.029583,"(Coffee, Hot chocolate)",2
21,0.029160,"(Bread, Pastry)",2
25,0.028209,"(Coffee, Cookies)",2
22,0.028104,"(Bread, Tea)",2


### <span style="color:orange"> <strong> 3-2. 최소 지지도를 바꿔보면? <strong> </span>

<span style="color:black; background-color:#FFF099; padding:2px 4px; border-radius:4px">
<strong> 🤓 min_support를 낮추면 어떤 일이 벌어지는지 눈으로 확인해봅시다! <strong>
</span>

In [14]:
for ms in [0.10, 0.05, 0.02, 0.01, 0.005, 0.001]:
    n = len(apriori(basket_df, min_support=ms, use_colnames=True))
    print(f"min_support = {ms:<6} →  빈발 항목 집합 {n:>4}개")

min_support = 0.1    →  빈발 항목 집합    4개
min_support = 0.05   →  빈발 항목 집합   11개
min_support = 0.02   →  빈발 항목 집합   33개
min_support = 0.01   →  빈발 항목 집합   61개
min_support = 0.005  →  빈발 항목 집합  114개
min_support = 0.001  →  빈발 항목 집합  471개


## **4. 연관 규칙 도출과 3대 지표 해석**

### <span style="color:orange"> <strong> 4-1. 규칙 만들기 <strong> </span>

In [15]:
rules = association_rules(frequent_itemsets,
                         metric="confidence",
                         min_threshold=0.2)

# 보기 편하게 핵심 컬럼만 남기고, frozenset을 문자열로 변환
rules_view = rules[['antecedents', 'consequents',
                    'antecedent support', 'consequent support',
                    'support', 'confidence', 'lift']].copy()
rules_view['antecedents'] = rules_view['antecedents'].apply(lambda x: ', '.join(x))
rules_view['consequents'] = rules_view['consequents'].apply(lambda x: ', '.join(x))

print("도출된 규칙 개수 :", len(rules_view))
rules_view

도출된 규칙 개수 : 13


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift
0,Cake,Bread,0.103856,0.327205,0.023349,0.224822,0.687097
1,Bread,Coffee,0.327205,0.478394,0.090016,0.275105,0.575059
2,Pastry,Bread,0.086107,0.327205,0.029160,0.338650,1.034977
3,Cake,Coffee,0.103856,0.478394,0.054728,0.526958,1.101515
4,Cake,Tea,0.103856,0.142631,0.023772,0.228891,1.604781
5,Cookies,Coffee,0.054411,0.478394,0.028209,0.518447,1.083723
6,Hot chocolate,Coffee,0.058320,0.478394,0.029583,0.507246,1.060311
7,Juice,Coffee,0.038563,0.478394,0.020602,0.534247,1.116750
8,Medialuna,Coffee,0.061807,0.478394,0.035182,0.569231,1.189878
9,Pastry,Coffee,0.086107,0.478394,0.047544,0.552147,1.154168


<span style="color:black; background-color:#FFD6E0; padding:2px 4px; border-radius:4px">
<strong> ✍️ Q6. `antecedent support`, `consequent support`, `support`는 각각 무엇인가요? <strong>
</span>

세 컬럼 모두 이름에 'support'가 들어가지만 의미가 전부 다릅니다.
각각을 확률 기호($P(A)$, $P(B)$, $P(A \cap B)$ 중 하나)와 연결하고, 한 문장씩 설명해보세요.

| 컬럼 | 확률 기호 | 설명 |
| --- | --- | --- |
| `antecedent support` |  |  |
| `consequent support` |  |  |
| `support` |  |  |

---

**📝 나의 답변**

>
>
>
>
>
>

### <span style="color:orange"> <strong> 4-2. ⚠️ 함정 발견하기 <strong> </span>

<span style="color:black; background-color:#FFF099; padding:2px 4px; border-radius:4px">
<strong> 🤓 자, 지지도가 가장 높은 규칙부터 봅시다. 가장 자주 함께 팔린 조합이니까 당연히 최고의 규칙이겠죠? <strong>
</span>

In [16]:
rules_view.sort_values('support', ascending=False).head(5)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift
1,Bread,Coffee,0.327205,0.478394,0.090016,0.275105,0.575059
3,Cake,Coffee,0.103856,0.478394,0.054728,0.526958,1.101515
11,Tea,Coffee,0.142631,0.478394,0.049868,0.349630,0.730840
9,Pastry,Coffee,0.086107,0.478394,0.047544,0.552147,1.154168
10,Sandwich,Coffee,0.071844,0.478394,0.038246,0.532353,1.112792


<span style="color:black; background-color:#FFD6E0; padding:2px 4px; border-radius:4px">
<strong> ✍️ Q7. 지지도 1위 규칙의 향상도(lift)를 보세요. 뭔가 이상하지 않나요? <strong>
</span>

지지도가 가장 높은 규칙은 `Bread → Coffee` 입니다. 이 가게에서 가장 자주 함께 팔린 조합이죠.
그런데 이 규칙의 **lift 값**을 확인해보세요.

- lift가 1보다 큰가요, 작은가요?
- 그 값은 무엇을 의미하나요? (Bread를 산 손님은 Coffee를 **더** 사나요, **덜** 사나요?)
- `consequent support`(= Coffee의 전체 구매율)와 `confidence`를 직접 비교해보세요. 어느 쪽이 더 큰가요?
- **이 함정이 우리에게 알려주는 교훈**을 한 문장으로 정리해보세요.

---

**📝 나의 답변**

>
>
>
>
>
>
>
>

### <span style="color:orange"> <strong> 4-3. 진짜 좋은 규칙 찾기 <strong> </span>

In [17]:
rules_view.sort_values('lift', ascending=False).head(8)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift
4,Cake,Tea,0.103856,0.142631,0.023772,0.228891,1.604781
12,Toast,Coffee,0.033597,0.478394,0.023666,0.704403,1.472431
8,Medialuna,Coffee,0.061807,0.478394,0.035182,0.569231,1.189878
9,Pastry,Coffee,0.086107,0.478394,0.047544,0.552147,1.154168
7,Juice,Coffee,0.038563,0.478394,0.020602,0.534247,1.116750
10,Sandwich,Coffee,0.071844,0.478394,0.038246,0.532353,1.112792
3,Cake,Coffee,0.103856,0.478394,0.054728,0.526958,1.101515
5,Cookies,Coffee,0.054411,0.478394,0.028209,0.518447,1.083723


<span style="color:black; background-color:#FFD6E0; padding:2px 4px; border-radius:4px">
<strong> ✍️ Q8. 향상도 1위와 2위 규칙을 문장으로 해석해보세요. <strong>
</span>

아래 빈칸을 채워서 완성된 문장을 만들어보세요. (숫자는 표에서 직접 읽어오세요)

**① Toast → Coffee**
> Toast를 구매한 손님 중 ______%가 Coffee도 함께 구매했으며,
> 이는 Coffee의 일반 구매율(______%)보다 ______배 높은 수치이다.
> 따라서 이 규칙은 ______(유의미하다 / 무의미하다).

**② Cake → Tea**
> Cake를 구매한 손님 중 ______%가 Tea도 함께 구매했으며,
> 이는 Tea의 일반 구매율(______%)보다 ______배 높은 수치이다.



---

**📝 나의 답변**

>
>
>
>
>
>
>
>
>
>

### <span style="color:orange"> <strong> 4-4. 음의 연관관계 <strong> </span>

In [18]:
# Tea와 Coffee 사이의 관계만 뽑아보기
rules_view[
    (rules_view['antecedents'].isin(['Tea', 'Coffee'])) &
    (rules_view['consequents'].isin(['Tea', 'Coffee']))
]

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift
11,Tea,Coffee,0.142631,0.478394,0.049868,0.34963,0.73084


<span style="color:black; background-color:#FFD6E0; padding:2px 4px; border-radius:4px">
<strong> ✍️ Q9. Tea와 Coffee의 lift는 왜 1보다 작을까요? <strong>
</span>

- 이 결과를 **손님의 행동**으로 설명해보세요. (왜 이런 패턴이 나타날까요?)
- 이 가게 사장님이 `"Tea 사는 손님한테 Coffee 쿠폰을 뿌리자!"` 라고 한다면, 여러분은 뭐라고 조언하겠나요?
- lift < 1인 규칙은 **쓸모없는 규칙**일까요? 아니면 그 나름의 활용법이 있을까요?

---

**📝 나의 답변**

>
>
>
>
>
>

### <span style="color:orange"> <strong> 4-5. 방향을 바꾸면? <strong> </span>

In [19]:
# 같은 두 품목이지만 방향만 다른 두 규칙
all_rules = association_rules(frequent_itemsets, metric="lift", min_threshold=0)
all_rules['A'] = all_rules['antecedents'].apply(lambda x: ', '.join(x))
all_rules['B'] = all_rules['consequents'].apply(lambda x: ', '.join(x))

all_rules[
    all_rules['A'].isin(['Toast', 'Coffee']) &
    all_rules['B'].isin(['Toast', 'Coffee'])
][['A', 'B', 'support', 'confidence', 'lift']]

,A,B,support,confidence,lift
26,Toast,Coffee,0.023666,0.704403,1.472431
27,Coffee,Toast,0.023666,0.049470,1.472431


<span style="color:black; background-color:#FFD6E0; padding:2px 4px; border-radius:4px">
<strong> ✍️ Q10. `Toast → Coffee`와 `Coffee → Toast`를 비교해보세요. <strong>
</span>

- **신뢰도(confidence)** 는 두 방향이 같나요, 다른가요? 왜 그럴까요?
- **향상도(lift)** 는 두 방향이 같나요, 다른가요? 수식 $\dfrac{P(A \cap B)}{P(A)P(B)}$ 를 보고 이유를 설명해보세요.


---

**📝 나의 답변**

>
>
>
>
>
>
>
>

## **5. Apriori vs FP-Growth**

### <span style="color:orange"> <strong> 5-1. 결과가 정말 같을까? <strong> </span>

In [20]:
ap_result = apriori(basket_df, min_support=0.02, use_colnames=True)
fp_result = fpgrowth(basket_df, min_support=0.02, use_colnames=True)

print("Apriori   가 찾은 항목 집합 :", len(ap_result), "개")
print("FP-Growth 가 찾은 항목 집합 :", len(fp_result), "개")
print()
print("두 결과가 완전히 동일한가? →", set(ap_result['itemsets']) == set(fp_result['itemsets']))
print()
print("[Apriori 출력 순서]")
print(ap_result.head(5))
print()
print("[FP-Growth 출력 순서]")
print(fp_result.head(5))

Apriori   가 찾은 항목 집합 : 33 개
FP-Growth 가 찾은 항목 집합 : 33 개

두 결과가 완전히 동일한가? → True

[Apriori 출력 순서]
    support     itemsets
0  0.036344  (Alfajores)
1  0.327205      (Bread)
2  0.040042    (Brownie)
3  0.103856       (Cake)
4  0.478394     (Coffee)

[FP-Growth 출력 순서]
    support         itemsets
0  0.327205          (Bread)
1  0.029054   (Scandinavian)
2  0.058320  (Hot chocolate)
3  0.054411        (Cookies)
4  0.038457         (Muffin)


### <span style="color:orange"> <strong> 5-2. 속도 비교 <strong> </span>

In [21]:
print(f"{'min_support':>12} | {'항목집합':>7} | {'Apriori':>9} | {'FP-Growth':>10} | 배율")
print("-" * 62)

for ms in [0.05, 0.01, 0.001, 0.0005, 0.0002, 0.0001]:
    t0 = time.time(); a = apriori(basket_df, min_support=ms, use_colnames=True)
    t_ap = time.time() - t0

    t0 = time.time(); f = fpgrowth(basket_df, min_support=ms, use_colnames=True)
    t_fp = time.time() - t0

    print(f"{ms:>12} | {len(a):>7} | {t_ap:>8.4f}s | {t_fp:>9.4f}s | {t_ap/t_fp:>4.1f}배")

 min_support |    항목집합 |   Apriori |  FP-Growth | 배율
--------------------------------------------------------------
        0.05 |      11 |   0.0062s |    0.0425s |  0.1배
        0.01 |      61 |   0.0365s |    0.0511s |  0.7배
       0.001 |     471 |   0.3739s |    0.0674s |  5.5배
      0.0005 |     927 |   0.9096s |    0.0848s | 10.7배
      0.0002 |    2673 |   2.9539s |    0.0926s | 31.9배
      0.0001 |   11200 |  17.0224s |    0.1692s | 100.6배


<span style="color:black; background-color:#FFD6E0; padding:2px 4px; border-radius:4px">
<strong> ✍️ Q11. FP-Growth는 항상 Apriori보다 빠른가요? <strong>
</span>

위 표를 위에서 아래로 훑어보세요. **배율** 컬럼의 변화가 핵심입니다.

- `min_support`가 **클 때**(0.05) 두 알고리즘 중 어느 쪽이 빨랐나요? 이론 수업에서 배운 내용과 같았나요?
- `min_support`가 **작아질수록** 배율은 어떻게 변하나요?
- 이 결과를 바탕으로, **"FP-Growth를 써야 하는 상황"** 과 **"Apriori로 충분한 상황"** 을 각각 정리해보세요.

<span style="color:gray">💡 힌트: FP-트리를 만드는 데도 초기 비용이 듭니다. 탐색할 조합이 적으면 그 비용을 회수하지 못하겠죠.</span>

---

**📝 나의 답변**

>
>
>
>
>
>
>
>

## **6. 종합 과제 : 사장님께 보고하기**

### <span style="color:orange"> <strong> 6-1. 최종 후보 규칙 선별 <strong> </span>

In [22]:
final_rules = rules_view[
    (rules_view['support'] >= 0.02) &      # 충분히 자주 발생하고
    (rules_view['confidence'] >= 0.5) &    # 예측이 어느 정도 맞으며
    (rules_view['lift'] > 1)               # 우연 이상의 연관이 있는
].sort_values('lift', ascending=False)

final_rules

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift
12,Toast,Coffee,0.033597,0.478394,0.023666,0.704403,1.472431
8,Medialuna,Coffee,0.061807,0.478394,0.035182,0.569231,1.189878
9,Pastry,Coffee,0.086107,0.478394,0.047544,0.552147,1.154168
7,Juice,Coffee,0.038563,0.478394,0.020602,0.534247,1.116750
10,Sandwich,Coffee,0.071844,0.478394,0.038246,0.532353,1.112792
3,Cake,Coffee,0.103856,0.478394,0.054728,0.526958,1.101515
5,Cookies,Coffee,0.054411,0.478394,0.028209,0.518447,1.083723
6,Hot chocolate,Coffee,0.058320,0.478394,0.029583,0.507246,1.060311


<span style="color:black; background-color:#FFD6E0; padding:2px 4px; border-radius:4px">
<strong> ✍️ Q13. 여러분이 이 베이커리의 데이터 분석가라면, 사장님께 무엇을 제안하시겠습니까? <strong>
</span>

지금까지의 분석 결과를 근거로 **구체적인 실행 방안 2가지**를 제안해보세요.
아래 항목을 반드시 포함해서 작성합니다.

1. **제안하는 액션** (세트메뉴 / 진열 위치 / 쿠폰 / 시간대 프로모션 등 자유)
2. **근거가 되는 규칙과 세 지표의 수치** (지지도 · 신뢰도 · 향상도를 모두 인용할 것)
3. **왜 `Bread → Coffee`(지지도 1위)는 제안하지 않았는지**에 대한 설명

<span style="color:gray">💡 "숫자가 높아서요"가 아니라 "이 숫자가 이런 의미이기 때문에"로 쓰는 게 핵심입니다.</span>

---

**📝 나의 답변**

>
>
>
>
>
>
>
>
>
>
>
>
>
>

---

### <span style="color:purple"> <strong> 🌟 도전 과제 (선택) <strong> </span>

여유가 있다면 아래 중 하나를 직접 코드로 확인해보세요.

1. `Date`와 `Time` 컬럼을 활용해 **오전(11시 이전) 장바구니**와 **오후 장바구니**를 나눠
   각각 연관 분석을 돌려보고, 규칙이 어떻게 달라지는지 비교해보기
2. 상위 20개 품목만 남기고 분석했을 때 규칙이 어떻게 달라지는지 확인해보기
3. `leverage`, `conviction`, `zhangs_metric` 컬럼이 각각 무엇을 의미하는지 찾아보고,
   `lift`와 어떻게 다른지 정리해보기

In [23]:
# 🌟 도전 과제용 셀


# **🤓 수고하셨습니다~**